<a href="https://colab.research.google.com/github/jc7qx/mcu2026-mlai-hw604/blob/main/imbclm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# make dataset
from sklearn.datasets import make_classification
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report



In [ ]:
# make dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=2,
                           n_redundant=10, n_clusters_per_class=1,
                           weights=[0.95], flip_y=0, random_state=1)

print(f'Original dataset shape {Counter(y)}')


Original dataset shape Counter({np.int64(0): 950, np.int64(1): 50})


In [ ]:
# create models
from sklearn.ensemble import RandomForestClassifier

#spplit data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

# base model
rfc_base = RandomForestClassifier(n_estimators=100, random_state=42)
rfc_base.fit(X_train, y_train)

# eval base model
y_pred = rfc_base.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

rpt = classification_report(y_test, y_pred, target_names=["0","1"])
print("\nClassification Report:\n", rpt)




Confusion Matrix:
 [[189   1]
 [  3   7]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99       190
           1       0.88      0.70      0.78        10

    accuracy                           0.98       200
   macro avg       0.93      0.85      0.88       200
weighted avg       0.98      0.98      0.98       200



In [ ]:
# balanced model
rfc_balanced = RandomForestClassifier(n_estimators=100, class_weight = "balanced", random_state=33)
rfc_balanced.fit(X_train, y_train)

y_pred_bal = rfc_balanced.predict(X_test)

cm = confusion_matrix(y_test, y_pred_bal)
print("Confusion Matrix:\n", cm)

rpt = classification_report(y_test, y_pred_bal, target_names=["0","1"])
print("\nClassification Report:\n", rpt)

Confusion Matrix:
 [[189   1]
 [  3   7]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99       190
           1       0.88      0.70      0.78        10

    accuracy                           0.98       200
   macro avg       0.93      0.85      0.88       200
weighted avg       0.98      0.98      0.98       200



In [ ]:
# make SMOTE dataset
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

print(f'Resampled dataset shape {Counter(y_res)}')

# SMOTE model
X_train_res, X_test_res, y_train_res, y_test_res = train_test_split(X_res, y_res, test_size=0.2, random_state=42)
rfc_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rfc_smote.fit(X_train_res, y_train_res)

# eval smote model
y_pred_smote = rfc_smote.predict(X_test)

cm = confusion_matrix(y_test, y_pred_smote)
print("Confusion Matrix:\n", cm)

rpt = classification_report(y_test, y_pred_smote, target_names=["0","1"])
print("\nClassification Report:\n", rpt)

Resampled dataset shape Counter({np.int64(0): 950, np.int64(1): 950})
Confusion Matrix:
 [[189   1]
 [  0  10]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00       190
           1       0.91      1.00      0.95        10

    accuracy                           0.99       200
   macro avg       0.95      1.00      0.97       200
weighted avg       1.00      0.99      1.00       200



# 交叉驗證方法

### 5-Fold Cross-Validation for the SMOTE Model

To evaluate the stability and generalization performance of the Random Forest model trained on SMOTE-resampled data, we'll perform 5-fold cross-validation. This will provide a more robust estimate of the model's performance than a single train-test split.

The cross-validation scores show the F1-score for each of the 5 folds. The mean F1 score gives an overall estimate of the model's performance, and the standard deviation indicates how much the performance varies across different data splits. A small standard deviation suggests a more stable model.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Re-instantiate the RandomForestClassifier with the same parameters as rfc_smote
# This ensures a fresh model is trained in each cross-validation fold.
rfc_cv = RandomForestClassifier(n_estimators=100, random_state=42)

# Perform 5-fold cross-validation on the SMOTE-resampled data (X_res, y_res)
# Using 'f1' as the scoring metric for imbalanced classes.
cv_scores = cross_val_score(rfc_cv, X_train, y_train, cv=5, scoring='f1')

print("Cross-validation F1 scores:", cv_scores)
print(f"Mean F1 score: {cv_scores.mean():.4f}")
print(f"Standard deviation of F1 scores: {cv_scores.std():.4f}")

Cross-validation F1 scores: [0.93333333 0.85714286 0.76923077 0.85714286 0.82352941]
Mean F1 score: 0.8481
Standard deviation of F1 scores: 0.0534


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Re-instantiate the RandomForestClassifier with the same parameters as rfc_smote
# This ensures a fresh model is trained in each cross-validation fold.
rfc_bal_cv = RandomForestClassifier(n_estimators=100, class_weight="balanced",random_state=42)

# Perform 5-fold cross-validation on the SMOTE-resampled data (X_res, y_res)
# Using 'f1' as the scoring metric for imbalanced classes.
cv_scores = cross_val_score(rfc_bal_cv, X, y, cv=5, scoring='f1')

print("Cross-validation F1 scores:", cv_scores)
print(f"Mean F1 score: {cv_scores.mean():.4f}")
print(f"Standard deviation of F1 scores: {cv_scores.std():.4f}")

Cross-validation F1 scores: [0.94736842 0.9        0.94736842 0.84210526 0.75      ]
Mean F1 score: 0.8774
Standard deviation of F1 scores: 0.0745


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Re-instantiate the RandomForestClassifier with the same parameters as rfc_smote
# This ensures a fresh model is trained in each cross-validation fold.
rfc_smote_cv = RandomForestClassifier(n_estimators=100, random_state=42)

# Perform 5-fold cross-validation on the SMOTE-resampled data (X_res, y_res)
# Using 'f1' as the scoring metric for imbalanced classes.
cv_scores = cross_val_score(rfc_smote_cv, X_res, y_res, cv=5, scoring='f1')

print("Cross-validation F1 scores:", cv_scores)
print(f"Mean F1 score: {cv_scores.mean():.4f}")
print(f"Standard deviation of F1 scores: {cv_scores.std():.4f}")

Cross-validation F1 scores: [0.97637795 0.96296296 0.9816273  0.98172324 0.97354497]
Mean F1 score: 0.9752
Standard deviation of F1 scores: 0.0069


In [ ]:
from imblearn.over_sampling import RandomOverSampler
# define oversampling strategy
oversample = RandomOverSampler(sampling_strategy='minority')
# fit and apply the transform
X_over, y_over = oversample.fit_resample(X, y)
# summarize class distribution
print(Counter(y_over))

rfc_cv = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rfc_cv, X_over, y_over, cv=5, scoring='f1')

print("Cross-validation F1 scores:", cv_scores)
print(f"Mean F1 score: {cv_scores.mean():.4f}")
print(f"Standard deviation of F1 scores: {cv_scores.std():.4f}")

Counter({np.int64(0): 950, np.int64(1): 950})
Cross-validation F1 scores: [0.99737533 0.99737533 0.99737533 0.9947644  0.99737533]
Mean F1 score: 0.9969
Standard deviation of F1 scores: 0.0010


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

# define undersample strategy
undersample = RandomUnderSampler(sampling_strategy='majority')
# fit and apply the transform
X_under, y_under = undersample.fit_resample(X, y)
# summarize class distribution
print(Counter(y_under))

rfc_cv = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rfc_cv, X_under, y_under, cv=5, scoring='f1')

print("Cross-validation F1 scores:", cv_scores)
print(f"Mean F1 score: {cv_scores.mean():.4f}")
print(f"Standard deviation of F1 scores: {cv_scores.std():.4f}")

Counter({np.int64(0): 50, np.int64(1): 50})
Cross-validation F1 scores: [1.         0.86956522 0.94736842 0.9        0.82352941]
Mean F1 score: 0.9081
Standard deviation of F1 scores: 0.0611
